# COMP5122M – Data Science Coursework

## Project: Vibe Check - How Gen Z Really Feels About AI



## Team - Shyamin 

### Members: 

Durgam Naga Deeraj Reddy (pvcl0430) ; Harshini Gonela (jklt0901) ; Karuna Anil Gholap (lctl0641) ; Rehan Ali (fnkx0448)



## Group Data Science Analysis

## Project Overview
This project analyses the Pew Research Center's Spring 2025 Global Attitudes Survey to investigate public attitudes towards artificial intelligence across 24 countries.

Task 2 focuses on verifying whether the preprocessing and analysis results are accurate by comparing the group's calculated percentages for AI awareness (Q30 – ai_heard) and AI optimism/concern (Q31 – ai_cncexc) against the official Pew Research Center published topline figures for at least three countries. The teammate must calculate the survey response percentages, compare them with the published results, identify any differences and investigate possible reasons for discrepancies. This step establishes whether the dataset has been correctly cleaned and whether the analysis produces reliable results.

## Task 2: Validation against published figures

Performed by Karuna Anil Gholap (lctl0641)

### Importing Required Libraries

In [5]:
import pandas as pd
import numpy as np

### Load CSV File

In [6]:
# load the cleaned dataset and original dataset for validation
target_countries = [13, 14, 22]
country_names = {13: 'Australia', 14: 'India', 22: 'South Africa'}
df = pd.read_csv("Task1_Cleaned_Dataset.csv", low_memory=False)
raw_df = pd.read_csv(
    "../Resources/Pew_Original_Data/Pew Research Center Global Attitudes Spring 2025 Dataset CSV.csv",
    low_memory=False
)

for variable in ["country", "weight", "ai_heard", "ai_cncexc"]:
    raw_df[variable] = pd.to_numeric(raw_df[variable], errors="coerce")

print("Cleaned dataset dimensions:", df.shape)
print("Original dataset dimensions:", raw_df.shape)


Cleaned dataset dimensions: (28333, 486)
Original dataset dimensions: (28333, 483)


### Validate the original data against Pew's published topline

The original CSV is required because Task 1 has recoded DK/refused responses to missing.

In [7]:
# full published topline values from Pew Spring 2025 Topline, page 48
# code 8 and code 9 are combined into the published DK/Refused category
full_topline = {
    ("Q30", 13): {"A lot": 44, "A little": 53, "Nothing at all": 3, "DK/Refused": 0},
    ("Q30", 14): {"A lot": 14, "A little": 32, "Nothing at all": 35, "DK/Refused": 19},
    ("Q30", 22): {"A lot": 30, "A little": 31, "Nothing at all": 34, "DK/Refused": 6},
    ("Q31", 13): {"More excited": 13, "More concerned": 49, "Equally concerned and excited": 38, "DK/Refused": 0},
    ("Q31", 14): {"More excited": 16, "More concerned": 19, "Equally concerned and excited": 39, "DK/Refused": 26},
    ("Q31", 22): {"More excited": 18, "More concerned": 30, "Equally concerned and excited": 42, "DK/Refused": 10}
}

question_specs = {
    "Q30": {
        "variable": "ai_heard",
        "labels": {1: "A lot", 2: "A little", 3: "Nothing at all"},
        "valid_codes": [1, 2, 3],
        "all_codes": [1, 2, 3, 8, 9]
    },
    "Q31": {
        "variable": "ai_cncexc",
        "labels": {1: "More excited", 2: "More concerned", 3: "Equally concerned and excited"},
        "valid_codes": [1, 2, 3],
        "all_codes": [1, 2, 3, 8, 9]
    }
}

full_validation_rows = []
cleaning_check_rows = []

for question, spec in question_specs.items():
    variable = spec["variable"]

    for country_id in target_countries:
        raw_country = raw_df[raw_df["country"] == country_id].copy()
        cleaned_country = df[df["country"] == country_id].copy()

        # full raw base includes substantive answers plus DK/refused
        raw_full = raw_country[raw_country[variable].isin(spec["all_codes"])].copy()
        raw_full_weight = raw_full["weight"].sum()

        for code, label in spec["labels"].items():
            category_weight = raw_full.loc[raw_full[variable] == code, "weight"].sum()
            calculated = category_weight / raw_full_weight * 100
            published = full_topline[(question, country_id)][label]
            full_validation_rows.append({
                "Question": question,
                "Country": country_names[country_id],
                "Response category": label,
                "Raw weighted %": round(calculated, 1),
                "Pew topline %": published,
                "Difference (pp)": round(calculated - published, 1)
            })

        dk_weight = raw_full.loc[raw_full[variable].isin([8, 9]), "weight"].sum()
        dk_calculated = dk_weight / raw_full_weight * 100
        dk_published = full_topline[(question, country_id)]["DK/Refused"]
        full_validation_rows.append({
            "Question": question,
            "Country": country_names[country_id],
            "Response category": "DK/Refused",
            "Raw weighted %": round(dk_calculated, 1),
            "Pew topline %": dk_published,
            "Difference (pp)": round(dk_calculated - dk_published, 1)
        })

        # confirm that Task 1 did not alter substantive responses
        raw_valid = raw_country[raw_country[variable].isin(spec["valid_codes"])].copy()
        clean_valid = cleaned_country[cleaned_country[variable].isin(spec["valid_codes"])].copy()
        raw_denominator = raw_valid["weight"].sum()
        clean_denominator = clean_valid["weight"].sum()

        for code, label in spec["labels"].items():
            raw_pct = raw_valid.loc[raw_valid[variable] == code, "weight"].sum() / raw_denominator * 100
            clean_pct = clean_valid.loc[clean_valid[variable] == code, "weight"].sum() / clean_denominator * 100
            cleaning_check_rows.append({
                "Question": question,
                "Country": country_names[country_id],
                "Response category": label,
                "Raw valid-only %": raw_pct,
                "Cleaned valid-only %": clean_pct,
                "Absolute difference (pp)": abs(raw_pct - clean_pct)
            })

full_topline_validation = pd.DataFrame(full_validation_rows)
cleaning_consistency = pd.DataFrame(cleaning_check_rows)

print("FULL TOPLINE VALIDATION – raw data versus Pew (page 48)")
display(full_topline_validation)

print("CLEANING CONSISTENCY – raw valid responses versus cleaned valid responses")
display(cleaning_consistency.round(4))

max_full_difference = full_topline_validation["Difference (pp)"].abs().max()
max_cleaning_difference = cleaning_consistency["Absolute difference (pp)"].max()

print(f"Maximum absolute full-topline difference: {max_full_difference:.1f} pp")
print(f"Maximum raw-versus-cleaned substantive-response difference: {max_cleaning_difference:.10f} pp")

# full-topline values are rounded in Pew's report; differences up to 0.6 pp are consistent with rounding
assert max_full_difference <= 0.6, "A full-topline difference exceeds the expected rounding tolerance."
assert max_cleaning_difference < 1e-8, "Task 1 changed at least one substantive response percentage."


FULL TOPLINE VALIDATION – raw data versus Pew (page 48)


,Question,Country,Response category,Raw weighted %,Pew topline %,Difference (pp)
0,Q30,Australia,A lot,43.7,44,-0.3
1,Q30,Australia,A little,53.4,53,0.4
2,Q30,Australia,Nothing at all,2.8,3,-0.2
3,Q30,Australia,DK/Refused,0.1,0,0.1
4,Q30,India,A lot,14.1,14,0.1
5,Q30,India,A little,31.6,32,-0.4
6,Q30,India,Nothing at all,35.5,35,0.5
7,Q30,India,DK/Refused,18.8,19,-0.2
8,Q30,South Africa,A lot,29.6,30,-0.4
9,Q30,South Africa,A little,31.0,31,0.0


CLEANING CONSISTENCY – raw valid responses versus cleaned valid responses


,Question,Country,Response category,Raw valid-only %,Cleaned valid-only %,Absolute difference (pp)
0,Q30,Australia,A lot,43.7279,43.7279,0.0
1,Q30,Australia,A little,53.4901,53.4901,0.0
2,Q30,Australia,Nothing at all,2.7820,2.7820,0.0
3,Q30,India,A lot,17.3252,17.3252,0.0
4,Q30,India,A little,38.9463,38.9463,0.0
5,Q30,India,Nothing at all,43.7285,43.7285,0.0
6,Q30,South Africa,A lot,31.3942,31.3942,0.0
7,Q30,South Africa,A little,32.8873,32.8873,0.0
8,Q30,South Africa,Nothing at all,35.7185,35.7185,0.0
9,Q31,Australia,More excited,12.6762,12.6762,0.0


Maximum absolute full-topline difference: 0.5 pp
Maximum raw-versus-cleaned substantive-response difference: 0.0000000000 pp


### Interpretation and Validation Notes

- The primary validation uses the original raw CSV to reproduce the complete Q30/Q31 topline, including DK/refused (codes 8 and 9 combined), for Australia, India and South Africa.
- The raw weighted results are compared with Pew's published whole-number percentages on page 48; small differences are expected from rounding.
- The raw-versus-cleaned check compares codes 1–3 only. Matching percentages show that Task 1 removed non-response codes without changing substantive Q30/Q31 responses.
- The secondary valid-only comparison re-normalises Pew's three substantive categories to 100%, because the cleaned dataset excludes DK/refused.
- Pew's README states that 'weight' is the general-population weight and must be used for general-population attitudes.
- Source references: Pew Research Center, *Global Attitudes & Trends, Spring 2025 Topline*, p. 48; *Spring 2025 Data Info* README.